In [1]:
!pip install faiss-gpu-cu12


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import faiss

In [3]:
import numpy as np
import pandas as pd

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [5]:
!gdown https://drive.google.com/file/d/141BAMKzuPuwWcecaefevoSkziUrqkuO5/view?usp=sharing --fuzzy

zsh:1: no matches found: https://drive.google.com/file/d/141BAMKzuPuwWcecaefevoSkziUrqkuO5/view?usp=sharing


In [6]:
!gdown https://drive.google.com/file/d/1Qks6QsznD2JeFlg3VTw1PXNi20cpTvKf/view?usp=sharing --fuzzy

zsh:1: no matches found: https://drive.google.com/file/d/1Qks6QsznD2JeFlg3VTw1PXNi20cpTvKf/view?usp=sharing


In [7]:
orcs = pd.read_parquet('orcs.parquet')

In [8]:
df = pd.read_parquet('employees.parquet')

In [9]:
df.head()

,name,surname,fathername,gender,birthdate,inn,passport
0,жумавой,волоколамц*в,садагет оглы,м,1973-12-24,804926960301,7880355409
1,мубн,невмятуллин,"али,вич",м,1984-07-06,111068355926,1411814346
2,"ан,рей",карпоэ,вяч.славовеч,м,1962-03-28,617713534516,3428889207
3,олрга,рубцова,викторовна,ж,1961-10-13,530330958056,5497629378
4,сергей,леусенко,григорьевич,м,1998-09-10,872242822954,8988513519


In [10]:
!pip install vectorizers


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [11]:
from vectorizers import NgramVectorizer
from vectorizers.transformers import InformationWeightTransformer

from sklearn.pipeline import make_pipeline
from sklearn.decomposition import TruncatedSVD

In [12]:
def normalize_name(s: str) -> str:
    s = str(s) if s is not None else ""
    s = s.strip().lower()
    s = s.replace("ё", "е")
    return s

def to_char_tokens(s: str):
    s = normalize_name(s)
    s = s.replace(" ", "")
    s = s.replace("-", "")
    return list(s)

In [13]:
def vectorize_names(
    col: pd.Series,
    ngram_size=3,
    use_information_weight=True,
    svd_dim=None,
):
    sequences = col.map(to_char_tokens).tolist()

    steps = []
    steps.append(NgramVectorizer(ngram_size=ngram_size))

    if use_information_weight:
        steps.append(InformationWeightTransformer())

    if svd_dim is not None and svd_dim > 0:
        steps.append(TruncatedSVD(n_components=svd_dim))

    model = make_pipeline(*steps)

    X = model.fit_transform(sequences)

    return X, model

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from tqdm import tqdm


class RacistTransformer(TransformerMixin, BaseEstimator):
    def __init__(self):
        self.column_ngrams = {
            'name': 3,
            'surname': 3,
            'fathername': 3,
            'inn': 1,
            'passport': 1,
            'birthdate': 2
        }
        self.vecs = {
        }

    def fit(self, X, y=None):
        X.fillna('', inplace=True)
        for col, ngram in tqdm(self.column_ngrams.items()):
            _, vec = vectorize_names(X[col], ngram_size=ngram, svd_dim=100)
            self.vecs[col] = vec
        return self

    def transform(self, X):
        X.fillna('', inplace=True)
        dfs = []
        for col in self.column_ngrams:
            df = pd.DataFrame.sparse.from_spmatrix(
                self.vecs[col].transform(X[col]))
            dfs.append(df)
        return pd.concat(dfs, axis='columns')

In [15]:
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

In [16]:
dim = 100

In [17]:
model = Pipeline([
    ('vec', RacistTransformer()),
    ('compress', PCA(n_components=dim))
])

In [ ]:
df_vec = model.fit_transform(df)

  0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
orcs_vec = model.transform(orcs)

/home/i3alumba/Projects/AI/ML/.venv/lib/python3.11/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [ ]:
# df_vec = df_vec.astype(np.float32)

: 

In [ ]:
# orcs_vec = orcs_vec.astype(np.float32)

In [ ]:
df_vec.shape, orcs_vec.shape

((2011759, 115461), (47633, 115461))

In [ ]:
n_centroids = int(np.sqrt(len(df_vec)))

In [22]:
index = faiss.index_factory(dim, f"IVF{n_centroids},Flat")

In [ ]:
res = faiss.StandardGpuResources() 
index = faiss.index_cpu_to_gpu(res, 0, index)

AttributeError: module 'faiss' has no attribute 'StandardGpuResources'

In [23]:
index.train(df_vec)

AssertionError: 

In [ ]:
index.add(df_vec)

In [ ]:
topn = 1
D, I = index.search(orcs_vec, topn)

In [ ]:
I.shape

(47633, 1)

In [ ]:
len(set(I[:, 0]))

44440

In [ ]:
sub = pd.DataFrame({
    'id': np.arange(0, len(I), 1),
    'orig_index': I[:, 0]
})

sub.to_parquet('sub.parquet')